## A Neural Network Regression Model for Black-Scholes Option Pricing

In this notebook we will use the option pricing dataset generated in the notebook <code>Generate Option Training Data.ipynb</code> to train a Neural Network model. 

In [ ]:
import pandas as pd
import numpy as np
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load Data

The data was saved to a csv file and so this is loaded into a Pandas dataframe.

In [ ]:
filename = 'OptionData.csv'
df = pd.read_csv(filename)
df.head()

### Split Data

The data can now be split into test and training sets. Given the data was randomly generated and that each row of the dataframe contains both put and call we can simply slide the dataframe. We will used 20,000 rows as the test set.

In [ ]:
train_df = df[:-20000]
test_df = df[-20000:]

### Prepare Data

The option data from the file is not in the right format for training. In particular all the data for a call and a put with the same inputs are on the same row of the dataframe and the frame contains the option "Greeks". The model will be trained to price both puts and calls so we need to split the data and add a binary input flag to show whether the example is for a put or call. Finally we need to split the input data (X) from the labels (y), the option value. The following function contains the all the steps that are required.

In [ ]:
def prepareData(data_df):
    drop_list = ['CallValue', 'CallDelta', 
                'CallGamma', 'CallTheta', 'CallVega', 
                'CallRho', 'PutValue', 'PutDelta', 
                'PutGamma', 'PutTheta', 'PutVega', 'PutRho']

    input_data = data_df.copy(deep=True)
    input_data = input_data.drop(drop_list, axis=1)
    
    call_input_data = input_data.copy(deep=True)
    put_input_data = input_data.copy(deep=True)
    
    call_input_data['PutCall'] = 1
    put_input_data['PutCall'] = 0
    
    frames_list = [call_input_data, put_input_data]
    X = pd.concat(frames_list)
    
    label_call = data_df['CallValue'].tolist()
    label_put = data_df['PutValue'].tolist()
    
    y = label_call + label_put
    y = np.array(y)
    
    return X, y

In [ ]:
X, y = prepareData(train_df)

In [ ]:
X_test, y_test = prepareData(test_df)

### Scale Data

Given the three real-valued inputs all have quite different scales it is appropriate the rescale them so they all have similar ranges. To do this we will use the <code>StandardScalar</code> from SciKit learn. The training data only is used the generate the scaling parameters and then these are applied to the test set, which must be kept independent.

In [ ]:
from sklearn.preprocessing import StandardScaler
standard_scalar = StandardScaler()
X[['Vol', 'Spot', 'T']] = standard_scalar.fit_transform(X[['Vol', 'Spot', 'T']])

As can be seen below, the parameters are now all scaled to have mean 0.0 and variance 1.0.

In [ ]:
X.head()

In [ ]:
X_test[['Vol', 'Spot', 'T']] = standard_scalar.transform(X_test[['Vol', 'Spot', 'T']])

### Shuffle Training Data

Given all the all the call values are listed first in the dataframe it is prudent to shuffle the traing examples before use.

In [ ]:
from sklearn.utils import shuffle
X, y = shuffle(X, y, random_state=42)

### Build Model

The Neural Network will have just two hidden layers with 50 neurons each using the ReLU activation function. Given this is a regression problem the output later is linear. In fact given we know this is an option price and it is always positive we could also use ReLU for the output. To train the model we will use the rmsprop optimizer and the loss function is *Mean Squared Error* which is appropriate for regression problems

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.fc1 = nn.Linear(4, 50)
        self.fc2 = nn.Linear(50, 50)
        self.fc3 = nn.Linear(50, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
model = NeuralNetwork().to(device)

### Train

The model will be trained through 50 epochs with a batch size of 10,000. This is relatively large but reflects the fact that the number of inputs is small and the training set itself is large. On this occasion only the loss will be monitored during training.

In [ ]:
optimizer = optim.RMSprop(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

In [ ]:
train_x = torch.Tensor(X.to_numpy()).to(device)
train_y = torch.Tensor(y).to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=10000, 
                                           shuffle=True, drop_last=True)

test_x = torch.Tensor(X_test.to_numpy()).to(device)
test_y = torch.Tensor(y_test).to(device)
test_dataset = TensorDataset(test_x, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=1000, 
                                          shuffle=False, drop_last=True)

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0

        # Training
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)
        train_errors.append(train_loss)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)

        test_loss /= len(test_loader.dataset)
        test_errors.append(test_loss)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    return history

In [ ]:
epochs = 50
history_dict = train_model(model, train_loader, test_loader,
                           loss_fn, optimizer, epochs)

### Plot the training loss

Next we can plot the training loss that was captured at the end of each epoch using matplotlib.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [ ]:
epochsrng = range(1, epochs + 1)

In [ ]:
plt.figure(figsize=(7,5)) 
    
loss_values = history_dict['train_loss']
test_loss_values = history_dict['test_loss']
    
plt.plot(epochsrng, loss_values, color='k', linestyle='-', label = 'Training Loss')
plt.plot(epochsrng, test_loss_values, color='k', linestyle='--', label = 'Test Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.ylim((0,20))
plt.savefig('BlackScholesBasic.png', dpi=300, bbox_inches='tight')